# Noted and MLOps Worklow Project

## 1. Introduction and Project Objectives
### 1.1 Introduction

This project addresses a time series forecasting problem based on the Jena Climate dataset, with the goal of predicting future air temperature from historical meteorological observations. More specifically, the task consists of a multivariate-input, univariate-output, multi-step forecasting problem, where a sequence of past weather measurements is used to forecast the air temperature for the next 24 hours. The underlying modelling problem was selected because it offers a realistic and sufficiently rich scenario to demonstrate the complete engineering lifecycle of an intelligent model, including data ingestion, preprocessing, experiment tracking, configuration control, automated orchestration, model registration, and serving.

The project builds upon previous work developed in the context of time series modelling, extending it into a production-oriented MLOps system. While earlier experimentation was mainly centred on model design and performance improvement, the present work reframes the forecasting task as a full engineering pipeline. The objective is therefore not only to obtain a competitive forecasting model, but also to ensure that every stage of the process is reproducible, modular, traceable, and deployable.

### 1.2 Project Motivation

Weather forecasting constitutes a highly relevant application domain for time series modelling due to its temporal dependency structure, multivariate nature, and practical interpretability. Air temperature in particular is influenced by multiple atmospheric variables, such as pressure, humidity, and wind conditions, making it a suitable target for studying sequential learning methods. From an engineering perspective, this problem is especially appropriate for an MLOps project because it combines several essential challenges: handling time-indexed data, enforcing leakage-free preprocessing, registering experiments consistently, managing evolving configurations, and exposing trained models through an API for real-time predictions. 

### 1.3 Objectives

The main objective of this project is to design and implement a complete end-to-end MLOps pipeline for multi-step air temperature forecasting using historical weather observations from the Jena Climate dataset.

More specifically, the project aims to:

- define a clear and modular repository structure suitable for MLOps workflows;
- ingest and version the dataset using reproducible data management practices;
- ensure strict data lineage through the integration of DVC and MLflow;
- manage model, data, and execution parameters dynamically using Hydra;
- automate the workflow through an Apache Airflow DAG covering data ingestion, preprocessing, training, and evaluation;
- register the best-performing forecasting model in the MLflow Model Registry;
- deploy the selected model through a FastAPI serving layer with REST endpoints;
- provide a minimal frontend interface for user interaction and real-time prediction;
- demonstrate the full workflow, from configuration updates to retraining, registration, and API-based inference.

## 2. Dataset and Problem Definition
### 2.1 Dataset Overview

This project uses the Jena Climate dataset, which contains meteorological observations collected in Jena, Germany, from January 2009 to December 2016. The original data is recorded at 10-minute intervals and includes several atmospheric variables, making it suitable for multivariate time series forecasting.

For this project, the dataset is loaded from a local CSV file and incorporated into the pipeline as a versioned data asset, ensuring reproducibility and traceability across experiments.

### 2.2 Data Preparation

An initial inspection was performed to validate the dataset structure, temporal consistency, and data quality. Duplicate observations were removed, and the data was ordered chronologically to preserve the integrity of the time series.

Since the original 10-minute frequency was unnecessarily granular for the forecasting objective, the data was resampled to hourly frequency using mean aggregation. This reduced noise, lowered computational cost, and aligned the dataset with the selected prediction horizon.

After resampling, a small number of missing values were identified and removed. The final modelling dataset consists of approximately 70,000 hourly observations, providing a long and regular time series for training and evaluation.

### 2.3 Forecasting Task

The problem is defined as a multivariate-input, univariate-output, multi-step forecasting task. The model uses historical weather observations to predict air temperature for the next 24 hours.

The target variable is T (degC), while the main input variables are:

T (degC) — air temperature
p (mbar) — atmospheric pressure
rh (%) — relative humidity
wv (m/s) — wind speed
max. wv (m/s) — maximum wind speed
wd (deg) — wind direction

These variables were selected because they capture meteorological conditions that directly influence short-term temperature dynamics.

### 2.4 Input–Output Formulation

Under the baseline configuration, the model uses a lookback window of 120 hours and predicts a 24-hour horizon. Thus, the system receives the previous five days of multivariate weather observations and outputs the expected temperature sequence for the following day.

This formulation is appropriate for sequence-based forecasting models and, from an engineering perspective, supports the implementation of a reproducible MLOps pipeline covering preprocessing, training, evaluation, and deployment.

### 2.5 Relevance to the MLOps Pipeline

This use case is well suited to the objectives of the project because it combines structured data ingestion, time-aware preprocessing, configurable training parameters, experiment tracking, and deployable inference. As such, it provides a realistic setting for demonstrating the full end-to-end MLOps lifecycle required in the project guidelines.

## 3. System Architecture and Project Structure
### 3.1 Overall System Architecture

The proposed solution follows a modular MLOps architecture organized across three repositories: JENA_WEATHER, JENA_CLIENT, and NOTED. This separation allows the forecasting pipeline, the user-facing client, and the supporting development platform to evolve independently while remaining integrated within the same end-to-end workflow.

The JENA_WEATHER repository contains the core machine learning pipeline. It includes dataset versioning with DVC, configuration files, Airflow DAG definitions, source code for data processing and training, and project artifacts related to the forecasting workflow. This repository represents the central component of the system, where the full lifecycle of the model is executed.

The JENA_CLIENT repository contains the client-side inference interface. Its role is to provide a lightweight frontend and backend layer capable of interacting with the deployed forecasting service and exposing predictions to end users.

The NOTED repository acts as an integrated support platform for development and system interaction. In addition to project management features, it provides access to operational components such as DVC, MLflow, Airflow, Hydra-related tooling, file inspection, and LLM-assisted interaction. Although it is not the central forecasting repository, it contributes to the engineering layer of the project by centralizing access to key MLOps services.

Overall, the architecture supports the complete workflow required by the project: data ingestion, preprocessing, feature engineering, training, evaluation, experiment tracking, model registration, promotion, and inference delivery.

### 3.2 Core Pipeline Structure

The main repository, JENA_WEATHER, is organized to support a reproducible and maintainable machine learning workflow. Its structure separates data assets, configuration files, orchestration logic, source code, notebooks, and reporting material into dedicated directories.

At the root level, the repository includes a .dvc directory for data versioning, a config folder for parameter management, a dags folder for Airflow orchestration, a data folder for dataset references, a models folder for trained artifacts, a src folder for the implementation of the pipeline, and auxiliary folders such as notebooks and report. This structure reflects a clear separation of concerns and supports the transition from experimentation to automated execution.

The configuration layer is managed through Hydra-style YAML files. The main configuration file defines default references for data, model, training, and scaling configurations, while specific subfolders contain parameter sets for each component. For example, the data configuration specifies the input file, resampling frequency, selected features, forecasting target, temporal lookback window, prediction horizon, and train/validation/test split. Model configurations define the neural architecture, including GRU layers, hidden units, regularization, optimizer, and loss function. Different scaler configurations are also kept separately, including min-max, robust, and standard scaling options.

This configuration design removes hardcoded values from the codebase and supports controlled experimentation through interchangeable parameter files, in line with the project requirement for dynamic configuration management.

### 3.3 Data Versioning and Storage Organization

The project uses DVC to version the dataset and manage large data artifacts outside the Git repository. The tracked dataset is referenced through a .dvc pointer file containing the file path, hash, and size metadata. This mechanism ensures that the exact version of the dataset can be recovered consistently across executions.

The DVC remote is configured over a MinIO S3-compatible bucket, allowing tracked data to be pushed and pulled from object storage. Within the broader platform, DVC operations are encapsulated through a dedicated management layer that supports initialization, file tracking, synchronization, version inspection, and status querying. This setup strengthens reproducibility and enforces strict data lineage by decoupling large files from source control while preserving version identity through hashes.

From a repository design perspective, this approach ensures that the machine learning pipeline remains lightweight at code level while still supporting robust access to versioned datasets and artifacts.

### 3.4 Airflow-Orchestrated Training Pipeline

The automated workflow is implemented through an Airflow DAG named jena_training_pipeline. This pipeline operationalizes the end-to-end training lifecycle and replaces manual execution with an orchestrated process.

The DAG includes the following main stages:

data ingestion, where the raw climate dataset is loaded, validated, and cleaned;
preprocessing, where the series is resampled to hourly frequency, features are selected, engineered variables are created, and temporal train/validation/test splits are produced;
data quality analysis, where an Evidently report is generated from the processed dataset;
model training, where the configured GRU model is prepared, trained, evaluated, and logged to MLflow;
model promotion, where the new run is registered and compared against the current champion before optional promotion;
data drift analysis, where the training and test distributions are compared through Evidently.

This DAG demonstrates a full pipeline that goes beyond simple training execution by integrating quality validation, experiment logging, registry operations, and monitoring-oriented reporting within the same workflow. This is strongly aligned with the engineering orientation of the project.

### 3.5 Experiment Tracking and Model Lifecycle Integration

The training stage is integrated with MLflow, which is used to register experiments, log parameters, store training metrics, and save the trained TensorFlow model. During execution, the pipeline records configuration-related metadata such as model type, model configuration, scaler type, lookback window, horizon, number of epochs, batch size, and seed. It also stores evaluation metrics in both scaled space and original temperature scale.

After training, the resulting run is passed to a promotion step that registers the candidate model under a named entry in the model registry and automatically promotes it if it outperforms the current production model according to the selected evaluation criterion. This design supports a structured model lifecycle and introduces a deployment-oriented decision layer directly into the pipeline.

### 3.6 Engineering Rationale

The overall project structure was designed to prioritize modularity, reproducibility, and automation. Separating the machine learning pipeline, the user-facing client, and the supporting platform improves maintainability and reduces coupling between components. Within the main repository, the separation between configuration, orchestration, data, and source code allows each part of the workflow to be managed independently and updated without affecting the full system structure.

This organization also supports the evaluation criteria of the project, particularly those related to repository standards, clean code practices, reproducibility, configuration management, pipeline orchestration, and model serving. Instead of centering the implementation around a single experimental notebook, the project adopts a more production-oriented architecture consistent with MLOps principles and with the goals of the curricular unit.